Download data from [UK-based online retail](https://archive.ics.uci.edu/dataset/502/online+retail+ii).
future is 369 days

## Importing Libraries and data

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
#from tqdm import tqdm
#import re
#import os
#from tqdm.notebook import tqdm
from tqdm.auto import tqdm
from collections import defaultdict

#from sentence_transformers import SentenceTransformer
#from sklearn.cluster import KMeans,MiniBatchKMeans, AgglomerativeClustering
#from sklearn.metrics import silhouette_score

In [2]:
#os.getcwd()
#df_2010 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2009-2010")
#df_2011 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2010-2011")
#df = pd.concat([df_2010, df_2011], ignore_index=True)

In [3]:
#df.to_csv("../data/raw/online_retail.csv", index=False)
df_full = pd.read_csv("../data/raw/online_retail.csv")

In [4]:
df_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [5]:
df_full['Invoice'] = df_full['Invoice'].astype(str).str.upper().str.strip()
df_full['InvoiceDate'] = pd.to_datetime(df_full['InvoiceDate'])
df_full = df_full.rename(columns={"Customer ID": "Customer_ID"})
df_full["Customer_ID"] = df_full["Customer_ID"].astype("Int64")
df_full['StockCode'] = df_full['StockCode'].astype(str).str.upper().str.strip()
df_full['Description'] = df_full['Description'].astype(str).str.upper().str.strip()
df_full['Country'] = df_full['Country'].astype(str).str.upper().str.strip()

df_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer_ID  824364 non-null   Int64         
 7   Country      1067371 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 66.2 MB


## Splitting the data into calibration and holdout periods

Customer Lifetime Value (CLV) models should always be evaluated on future customer behavior rather than the same data used for model development. To accomplish this, the transaction history is divided into two non-overlapping time periods:

- **Calibration period:** Used to estimate customer behavior and fit the predictive models.
- **Holdout period:** Used only for evaluation. Transactions in this period are treated as unseen future data and are used to assess how well the models generalize.

In this project, **5 December 2010** is selected as the split date.

- Transactions **before 5 December 2010** belong to the **calibration period**.
- Transactions **on or after 5 December 2010** belong to the **holdout period**.

Performing this split immediately after loading the dataset ensures that all subsequent data cleaning, feature engineering, and model evaluation are conducted independently within each period. This avoids any information leakage from the future (holdout period) into the past (calibration period), resulting in a more realistic evaluation of customer lifetime value predictions.

In [6]:
# Split date
split_date = pd.Timestamp("2010-12-05")

# Calibration datasets
df = df_full[df_full["InvoiceDate"] <= split_date].copy()

print(f"Calibration transactions: {len(df):,}")
print(f"Holdout transactions   :  {len(df_full[df_full["InvoiceDate"] > split_date]):,}")

Calibration transactions: 517,776
Holdout transactions   :  549,595


# Identifying and handling cancelled/ returned order

This step assigns a clear transaction status to each line item by matching negative-quantity rows against earlier positive-quantity rows from the same customer, product, quantity, and price.

The idea is simple:

- A **positive** quantity usually means a normal purchase.
- A **negative** quantity usually means a return, cancellation, or reversal.
- If a negative row can be matched to a previous positive row with the same customer, stock code, absolute quantity, and price, then the two rows belong together.
- The positive row is marked as **`cancelled`**.
- The negative row is marked as **`return: purchase found`**.
- If no matching purchase exists, the negative row is marked as **`return: purchase not found`**.
- Any remaining positive rows that were never matched to a return are marked as **`completed`**.

This approach is much faster than filtering the whole dataframe for every negative row. Instead, the data is processed once in chronological order, and a hash-based lookup structure stores unmatched positive rows. That makes matching efficient even for large datasets.

The `status` column is a new descriptive field that summarizes the outcome of the matching logic for each row. It is useful later for cleaning, analysis, CLV modeling, and churn/retention work because it separates normal purchases from returns and cancellations.

In [7]:
df.info()

<class 'pandas.DataFrame'>
Index: 517776 entries, 0 to 532879
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      517776 non-null  str           
 1   StockCode    517776 non-null  str           
 2   Description  514871 non-null  str           
 3   Quantity     517776 non-null  int64         
 4   InvoiceDate  517776 non-null  datetime64[us]
 5   Price        517776 non-null  float64       
 6   Customer_ID  412989 non-null  Int64         
 7   Country      517776 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 36.0 MB


In [8]:
# Sort by time
df = df.sort_values("InvoiceDate").reset_index(drop=True)

# Status column
df["status"] = pd.NA

# ---------------------------------------------------
# Matching structure:
# key = (Customer_ID, StockCode, abs(Quantity), Price)
# value = list of unmatched positive row positions
# ---------------------------------------------------
open_sales = defaultdict(list)

# Use tqdm for progress tracking
for i, row in tqdm(
    enumerate(df.itertuples(index=False)),
    total=len(df),
    desc="Matching returns"
):
    key = (
        row.Customer_ID,
        row.StockCode,
        abs(row.Quantity),
        row.Price
    )
    if row.Quantity > 0:
        open_sales[key].append(i)

    elif row.Quantity < 0:
        if open_sales[key]:
            match_i = open_sales[key].pop()
            df.at[match_i, "status"] = "cancelled"
            df.at[i, "status"] = "return: purchase found"
        else:
            df.at[i, "status"] = "return: purchase not found"

# Any remaining positive rows are completed purchases
df.loc[(df["Quantity"] > 0) & (df["status"].isna()), "status"] = "completed"

Matching returns:   0%|          | 0/517776 [00:00<?, ?it/s]

In [9]:
df['status'] = df['status'].astype(str)
df.info()

## Keeping only completed orders
df = df[df['status']=='completed']
#df_return = df[df['status'] == 'return: purchase found']

print(f'Size of the calibration data after adjusting returned order is: {len(df)}')

<class 'pandas.DataFrame'>
RangeIndex: 517776 entries, 0 to 517775
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      517776 non-null  str           
 1   StockCode    517776 non-null  str           
 2   Description  514871 non-null  str           
 3   Quantity     517776 non-null  int64         
 4   InvoiceDate  517776 non-null  datetime64[us]
 5   Price        517776 non-null  float64       
 6   Customer_ID  412989 non-null  Int64         
 7   Country      517776 non-null  str           
 8   status       517776 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(5)
memory usage: 36.0 MB
Size of the calibration data after adjusting returned order is: 502378


In [10]:
# droping status column before saving to csv
df.drop(columns = 'status', inplace=True)
#df_return.drop(columns = 'status', inplace=True)


## BASIC CLEANING
Before moving to modeling, the dataset needs a few consistency checks so each transaction line can be trusted as a valid customer purchase record.

First, rows with missing `Customer_ID` are removed. These rows cannot be reliably linked to a customer, so they are not useful for customer-level analysis such as churn, CLV, or retention modeling.

Next, the relationship between `Customer_ID` and `Country` is checked. In a clean transactional dataset, the same customer should usually belong to one country. If a customer appears under multiple countries, the code identifies those cases and resolves them by assigning the most representative country for that customer. The chosen country is the one linked to the highest number of invoices, and if there is a tie, the one with the highest total quantity.

After that, the code checks whether each invoice belongs to only one customer. A single invoice should normally map to a single `Customer_ID`. If one invoice is linked to multiple customer IDs, those cases are flagged for review.

Finally, the `Description` column is standardized. The same `StockCode` should generally refer to the same product name, but inconsistent descriptions can appear in raw retail data. To fix this, the most common description for each `StockCode` is kept and assigned back to the dataframe.

These cleaning steps make the dataset more consistent and help ensure that later analyses are based on stable customer, product, and transaction definitions.

In [11]:
###  Checking if customer and country mismatches

# Creating new dataframe by dropping rows with blank customer_ID
rows_with_customer = df[df['Customer_ID'].notna()].copy()

# Count unique Country per Customer_ID
customer_country_check = (
    rows_with_customer
    .groupby('Customer_ID')
    .agg(
        n_countries=('Country', lambda x: x.nunique(dropna=False))
    )
    .reset_index()
)

# Customers linked to more than one country
country_mismatch = customer_country_check[
    customer_country_check['n_countries'] > 1
]

# Save all original rows belonging to those multiple country customers
customer_country_mismatches = (
    rows_with_customer[
        rows_with_customer['Customer_ID'].isin(country_mismatch['Customer_ID'])
    ]
    .sort_values(['Customer_ID', 'Country', 'InvoiceDate'])
)

if country_mismatch.shape[0] == 0:
    print('No mismatch between Customer_ID and Country')
else:
    print(f'❌ Country column of {country_mismatch.shape[0]} customers are not unique between different lines. \n \n ID\'s of the customres are:\n')
    print(customer_country_mismatches[
    ['Customer_ID', 'Country']
].drop_duplicates().sort_values(['Customer_ID', 'Country']).head(50))

❌ Country column of 5 customers are not unique between different lines. 
 
 ID's of the customres are:

        Customer_ID         Country
12491         12417         BELGIUM
365827        12417           SPAIN
39830         12422       AUSTRALIA
205288        12422     SWITZERLAND
467038        12423         BELGIUM
321987        12423         DENMARK
110889        12431       AUSTRALIA
57311         12431         BELGIUM
70978         12745            EIRE
198043        12745  UNITED KINGDOM


In [12]:
### Resolving mismatch between country and customer

# Invoice count and total quantity by Customer_ID
customer_country_stats = (
    rows_with_customer
    .groupby(['Customer_ID', 'Country'])
    .agg(
        n_invoices=('Invoice', 'nunique'),
        total_quantity=('Quantity', 'sum')
    )
    .reset_index()
)

# Pick mode country:
# 1. highest number of invoices
# 2. if tied, highest total quantity
mode_country_per_customer = (
    customer_country_stats
    .sort_values(
        ['Customer_ID', 'n_invoices', 'total_quantity'],
        ascending=[True, False, False]
    )
    .drop_duplicates('Customer_ID', keep='first')
    [['Customer_ID', 'Country']]
    .rename(columns={'Country': 'mode_country'})
)

# Add mode country to original df
df = df.merge(
    mode_country_per_customer,
    on='Customer_ID',
    how='left'
)

# Replace Country only for rows with given Customer_ID
df.loc[df['Customer_ID'].notna(), 'Country'] = df.loc[
    df['Customer_ID'].notna(),
    'mode_country'
]

# Drop helper column
df = df.drop(columns='mode_country')

print('Mismatches resolved between Customer_ID and Country, by updating country column. ✅')

Mismatches resolved between Customer_ID and Country, by updating country column. ✅


In [13]:
### Chacking whether rows corresponds to a same invoice referes to a same Customer_ID 

# Count unique Customer_ID  per Invoice
# dropna=False means missing Customer_ID is counted as its own value
invoice_check = (
    df.groupby('Invoice')
      .agg(
          n_customers=('Customer_ID', lambda x: x.nunique(dropna=False))
      )
      .reset_index()
)

# Invoices where same invoice has different customer 
bad_invoices = invoice_check[(invoice_check['n_customers'] > 1)]

# Save all original rows belonging to those problematic invoices
invoice_exceptions = (
    df[df['Invoice'].isin(bad_invoices['Invoice'])]
    .sort_values(['Invoice', 'Customer_ID'])
)


if bad_invoices.shape[0] == 0:
    print('No mismatch found between invoice and Customer_ID.  ✅')
else:
    print(f'{bad_invoices.shape[0]} mismatches observed between invoice and Customer_ID. Some mismatches as examples are:')
    print(invoice_exceptions[['Invoice', 'Customer_ID', 'Country']].drop_duplicates().head(5))

No mismatch found between invoice and Customer_ID.  ✅


In [14]:
### Updating description column
##  Same stock code should point to same items.

collapsed = (
    df.groupby(['StockCode', 'Description'])
      .size()
      .rename('count')
      .reset_index()
)

collapsed['rn'] = (
    collapsed.groupby('StockCode')['count']
             .rank(method='first', ascending=False)
)

collapsed = collapsed[collapsed['rn'] == 1]

df = df.merge(collapsed, on='StockCode', how='left', suffixes=('', '_top'))
df['Description'] = df['Description_top']
df = df.drop(columns=['Description_top', 'rn','count'])

print('Mismatches between description and customer ID is resolved by updating description column. ✅')

Mismatches between description and customer ID is resolved by updating description column. ✅


In [15]:
# Number of rows before removing rows with missing Customer_ID
print(f"Rows before removing rows with missing Customer_ID : {len(df)}")

# Remove rows with missing Customer_ID
df = df[df["Customer_ID"].notna()].copy()

print(f"Rows remaining                                     : {len(df)}")

Rows before removing rows with missing Customer_ID : 502378
Rows remaining                                     : 400249


In [16]:
# Number of duplicate rows (considering all columns)
n_duplicates = df.duplicated().sum()

if n_duplicates == 0:
    print("✅ No duplicate rows found.")
else:
    print(f"❌ Found {n_duplicates:,} duplicate rows.")

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)

    print(f"Deleted {n_duplicates:,} duplicate rows.")
    print(f"Remaining rows: {len(df):,}")

❌ Found 11,506 duplicate rows.
Deleted 11,506 duplicate rows.
Remaining rows: 388,743


In [17]:
# Total rows
n_rows = len(df)

# Number of unique (Invoice, StockCode) pairs
n_unique = df[['Invoice', 'StockCode']].drop_duplicates().shape[0]

print(f"Total rows: {n_rows:,}")
print(f"Unique (Invoice, StockCode) pairs: {n_unique:,}")

if n_rows == n_unique:
    print("✅ Every (Invoice, StockCode)  is unique.")
else:
    print(f"❌ Found {n_rows - n_unique:,} duplicate (Invoice, StockCode) pairs.")

Total rows: 388,743
Unique (Invoice, StockCode) pairs: 383,017
❌ Found 5,726 duplicate (Invoice, StockCode) pairs.


```text
(Invoice, StockCode) combination supposed be unique across the data. We found 5,726 duplicate (Invoice, StockCode) pairs.
 This behavior is expected in transaction-level retail data because an invoice may contain multiple line items for the same product due to separate entries, adjustments, or ERP order-line recording. Since customer value analysis is performed at the invoice level, these records will be retained and aggregated in the next steps.
```

In [18]:
df["line_amount"] = df["Quantity"] * df["Price"]

df = (
    df.groupby(["Invoice", "StockCode"], as_index=False)
      .agg(
          Quantity=("Quantity", "sum"),
          line_amount=("line_amount", "sum"),
          Description=("Description", "first"),
          InvoiceDate=("InvoiceDate", "first"),
          **{"Customer_ID": ("Customer_ID", "first")},
          Country=("Country", "first")
      )
)

df["Price"] = df["line_amount"] / df["Quantity"]

# Total rows
n_rows = len(df)

# Number of unique (Invoice, StockCode) pairs
n_unique = df[['Invoice', 'StockCode']].drop_duplicates().shape[0]

if n_rows == n_unique:
    print("After cleaning, (Invoice, StockCode)  is unique for every lines. ✅")
    print(f"Total lines after handling duplicates is: {n_unique:,}")
else:
    print(f"❌ Found {n_rows - n_unique:,} duplicate (Invoice, StockCode) pairs from total rows: {n_rows:,}.")

After cleaning, (Invoice, StockCode)  is unique for every lines. ✅
Total lines after handling duplicates is: 383,017


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 383017 entries, 0 to 383016
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      383017 non-null  str           
 1   StockCode    383017 non-null  str           
 2   Quantity     383017 non-null  int64         
 3   line_amount  383017 non-null  float64       
 4   Description  383017 non-null  str           
 5   InvoiceDate  383017 non-null  datetime64[us]
 6   Customer_ID  383017 non-null  Int64         
 7   Country      383017 non-null  str           
 8   Price        383017 non-null  float64       
dtypes: Int64(1), datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 26.7 MB


In [20]:
'''
# Customer-level totals
customer_price_stats = (
    df.groupby('Customer ID', as_index=False)
      .agg(
          total_quantity=('Quantity', 'sum'),
          total_line_amount=('line_amount', 'sum')
      )
)

# Average item price per customer
customer_price_stats['avg_item_price'] = (
    customer_price_stats['total_line_amount']
    / customer_price_stats['total_quantity']
)

# Customers to keep:
# avg_item_price > 0 and <= 1000
valid_customers = customer_price_stats.loc[
    (customer_price_stats['avg_item_price'] > 0) &
    (customer_price_stats['avg_item_price'] <= 100),
    'Customer ID'
]

# Filter the original dataframe
df = df[df['Customer ID'].isin(valid_customers)].copy()

print("Remaining rows:", len(df))
print("Remaining customers:", df['Customer ID'].nunique())

removed_customers = customer_price_stats.loc[
    (customer_price_stats['avg_item_price'] <= 0) |
    (customer_price_stats['avg_item_price'] > 100)
]

print("Customers removed:", len(removed_customers))
'''

'\n# Customer-level totals\ncustomer_price_stats = (\n    df.groupby(\'Customer ID\', as_index=False)\n      .agg(\n          total_quantity=(\'Quantity\', \'sum\'),\n          total_line_amount=(\'line_amount\', \'sum\')\n      )\n)\n\n# Average item price per customer\ncustomer_price_stats[\'avg_item_price\'] = (\n    customer_price_stats[\'total_line_amount\']\n    / customer_price_stats[\'total_quantity\']\n)\n\n# Customers to keep:\n# avg_item_price > 0 and <= 1000\nvalid_customers = customer_price_stats.loc[\n    (customer_price_stats[\'avg_item_price\'] > 0) &\n    (customer_price_stats[\'avg_item_price\'] <= 100),\n    \'Customer ID\'\n]\n\n# Filter the original dataframe\ndf = df[df[\'Customer ID\'].isin(valid_customers)].copy()\n\nprint("Remaining rows:", len(df))\nprint("Remaining customers:", df[\'Customer ID\'].nunique())\n\nremoved_customers = customer_price_stats.loc[\n    (customer_price_stats[\'avg_item_price\'] <= 0) |\n    (customer_price_stats[\'avg_item_price\'] > 

In [21]:
df.head(3)

,Invoice,StockCode,Quantity,line_amount,Description,InvoiceDate,Customer_ID,Country,Price
0,489434,21232,24,30.0,STRAWBERRY CERAMIC TRINKET BOX,2009-12-01 07:45:00,13085,UNITED KINGDOM,1.25
1,489434,21523,10,59.5,DOOR MAT FANCY FONT HOME SWEET HOME,2009-12-01 07:45:00,13085,UNITED KINGDOM,5.95
2,489434,21871,24,30.0,SAVE THE PLANET MUG,2009-12-01 07:45:00,13085,UNITED KINGDOM,1.25


## CREATING TABLES

In [22]:
## creating invoice table: Group by customer and invoice, then sum the costs and quantities
df_invoice = df.groupby(['Invoice', 'Customer_ID','InvoiceDate', 'Country']).agg(
    total_amount=('line_amount', 'sum'),
    number_of_items=('Quantity', 'sum')
).reset_index()

### Creating product table
df_product = (
    df.groupby('StockCode', as_index=False)['Description']
      .first()
)

df.drop(columns=['Customer_ID','InvoiceDate', 'Country', 'Description'], inplace=True)

In [23]:
# make a copy
df_order = df_invoice.copy()

# extract date only from date-time
df_order["order_date"] = df_order["InvoiceDate"].dt.date

# group same customer + same day into one order
df_order = (
    df_order
    .groupby(["Customer_ID", "order_date"], as_index=False)
    .agg(
        total_amount=("total_amount", "sum"),
        number_of_items=("number_of_items", "sum"),
        Country=("Country", "first"),
        InvoiceDate=('InvoiceDate', 'first')
    )
)

# create order_id as string: Customer_ID + date
df_order["order_id"] = (
    df_order["Customer_ID"].astype(str) + "_" +
    pd.to_datetime(df_order["order_date"]).dt.strftime("%Y-%m-%d")
)

df_order.drop(columns='InvoiceDate', inplace=True)

df_order['order_date'] = pd.to_datetime(df_order['order_date'])

In [24]:
df_order.info()

<class 'pandas.DataFrame'>
RangeIndex: 16501 entries, 0 to 16500
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype        
---  ------           --------------  -----        
 0   Customer_ID      16501 non-null  Int64        
 1   order_date       16501 non-null  datetime64[s]
 2   total_amount     16501 non-null  float64      
 3   number_of_items  16501 non-null  int64        
 4   Country          16501 non-null  str          
 5   order_id         16501 non-null  str          
dtypes: Int64(1), datetime64[s](1), float64(1), int64(1), str(2)
memory usage: 789.7 KB


In [25]:
df_order.to_csv('calib_order_table.csv')

# CLEANING HOLDOUT PERIOD

In [26]:
# Holdout datasets
df = df_full[df_full["InvoiceDate"] > split_date].copy()

In [27]:
# Sort by time
df = df.sort_values("InvoiceDate").reset_index(drop=True)

# Status column
df["status"] = pd.NA

# ---------------------------------------------------
# Matching structure:
# key = (Customer_ID, StockCode, abs(Quantity), Price)
# value = list of unmatched positive row positions
# ---------------------------------------------------
open_sales = defaultdict(list)

# Use tqdm for progress tracking
for i, row in tqdm(
    enumerate(df.itertuples(index=False)),
    total=len(df),
    desc="Matching returns"
):
    key = (
        row.Customer_ID,
        row.StockCode,
        abs(row.Quantity),
        row.Price
    )
    if row.Quantity > 0:
        open_sales[key].append(i)

    elif row.Quantity < 0:
        if open_sales[key]:
            match_i = open_sales[key].pop()
            df.at[match_i, "status"] = "cancelled"
            df.at[i, "status"] = "return: purchase found"
        else:
            df.at[i, "status"] = "return: purchase not found"

# Any remaining positive rows are completed purchases
df.loc[(df["Quantity"] > 0) & (df["status"].isna()), "status"] = "completed"

Matching returns:   0%|          | 0/549595 [00:00<?, ?it/s]

In [28]:
df['status'] = df['status'].astype(str)
df.info()

## Keeping only completed orders
df = df[df['status']=='completed']
#df_return = df[df['status'] == 'return: purchase found']

# droping status column before saving to csv
df.drop(columns = 'status', inplace=True)
#df_return.drop(columns = 'status', inplace=True)

<class 'pandas.DataFrame'>
RangeIndex: 549595 entries, 0 to 549594
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      549595 non-null  str           
 1   StockCode    549595 non-null  str           
 2   Description  548118 non-null  str           
 3   Quantity     549595 non-null  int64         
 4   InvoiceDate  549595 non-null  datetime64[us]
 5   Price        549595 non-null  float64       
 6   Customer_ID  411375 non-null  Int64         
 7   Country      549595 non-null  str           
 8   status       549595 non-null  str           
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), str(5)
memory usage: 38.3 MB


In [29]:
### Chacking whether rows corresponds to a same invoice referes to a same Customer_ID 

# Count unique Customer_ID  per Invoice
# dropna=False means missing Customer_ID is counted as its own value
invoice_check = (
    df.groupby('Invoice')
      .agg(
          n_customers=('Customer_ID', lambda x: x.nunique(dropna=False))
      )
      .reset_index()
)

# Invoices where same invoice has different customer 
bad_invoices = invoice_check[(invoice_check['n_customers'] > 1)]

# Save all original rows belonging to those problematic invoices
invoice_exceptions = (
    df[df['Invoice'].isin(bad_invoices['Invoice'])]
    .sort_values(['Invoice', 'Customer_ID'])
)


if bad_invoices.shape[0] == 0:
    print('No mismatch found between invoice and Customer_ID.  ✅')
else:
    print(f'{bad_invoices.shape[0]} mismatches observed between invoice and Customer_ID. Some mismatches as examples are:')
    print(invoice_exceptions[['Invoice', 'Customer_ID', 'Country']].drop_duplicates().head(5))

No mismatch found between invoice and Customer_ID.  ✅


In [30]:
# Number of duplicate rows (considering all columns)
n_duplicates = df.duplicated().sum()

if n_duplicates == 0:
    print("✅ No duplicate rows found.")
else:
    print(f"❌ Found {n_duplicates:,} duplicate rows.")

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)

    print(f"Deleted {n_duplicates:,} duplicate rows.")
    print(f"Remaining rows: {len(df):,}")

❌ Found 19,858 duplicate rows.
Deleted 19,858 duplicate rows.
Remaining rows: 516,029


In [31]:
df["line_amount"] = df["Quantity"] * df["Price"]

In [32]:
## creating invoice table: Group by customer and invoice, then sum the costs and quantities
df_invoice = df.groupby(['Invoice', 'Customer_ID','InvoiceDate', 'Country']).agg(
    total_amount=('line_amount', 'sum'),
    number_of_items=('Quantity', 'sum')
).reset_index()

In [33]:
# make a copy
df_order = df_invoice.copy()

# extract date only from date-time
df_order["order_date"] = df_order["InvoiceDate"].dt.date

# group same customer + same day into one order
df_order = (
    df_order
    .groupby(["Customer_ID", "order_date"], as_index=False)
    .agg(
        total_amount=("total_amount", "sum"),
        number_of_items=("number_of_items", "sum"),
        Country=("Country", "first"),
        InvoiceDate=('InvoiceDate', 'first')
    )
)

# create order_id as string: Customer_ID + date
df_order["order_id"] = (
    df_order["Customer_ID"].astype(str) + "_" +
    pd.to_datetime(df_order["order_date"]).dt.strftime("%Y-%m-%d")
)

df_order.drop(columns='InvoiceDate', inplace=True)

df_order['order_date'] = pd.to_datetime(df_order['order_date'])

In [34]:
df_order.info()

<class 'pandas.DataFrame'>
RangeIndex: 16443 entries, 0 to 16442
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype        
---  ------           --------------  -----        
 0   Customer_ID      16443 non-null  Int64        
 1   order_date       16443 non-null  datetime64[s]
 2   total_amount     16443 non-null  float64      
 3   number_of_items  16443 non-null  int64        
 4   Country          16443 non-null  str          
 5   order_id         16443 non-null  str          
dtypes: Int64(1), datetime64[s](1), float64(1), int64(1), str(2)
memory usage: 787.0 KB


In [35]:
df_order.to_csv('holdout_order_table.csv')